In [ ]:
import json
import math
import pathlib
import re

import geopy.distance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:

project_root = pathlib.Path('../../..')
project_root.resolve()

In [ ]:
metroflex_data = project_root / 'data/metroflex'
metroflex_data.resolve()

In [ ]:
with open(metroflex_data / 'metroflex-addresses.json', 'r') as f:
    address_locations = json.load(f)
address_locations

In [ ]:
def max_distance(matches):
    dist = 0
    loc1 = matches[0]['location']
    for match in matches[1:]:
        loc2 = match['location']
        dist = max(
            dist,
            geopy.distance.geodesic(
                (loc1['lat'], loc1['lng']),
                (loc2['lat'], loc2['lng']),
            ).m
        )
    return dist


def filter_by_state(matches, state):
    return [
        match
        for match in matches
        if match['address_components']['state'] == state
    ]


def filter_by_matching_start(matches, key):
    start = key.lower().split(' ')[0] + ' '
    positive_matches = [
        match
        for match in matches
        if match['formatted_address'].lower().startswith(start)
    ]
    if len(positive_matches) > 0:
        return positive_matches
    return matches


def filter_by_cities(matches, cities):
    return [
        match
        for match in matches
        if match['address_components']['city'] in cities
    ]

def filter_by_matching_suffixes(matches, key, suffix_pairs):
    for sfx, ptx in suffix_pairs:
        if re.search(ptx, key.lower()) is None:
            continue
        matches = [
            match
            for match in matches
            if match['address_components'].get('suffix', '').lower() == sfx
        ]
    return matches


def collapse_identical_addresses(matches):
    addys = {}
    for match in sorted(matches, key=lambda x: x['accuracy']):
        addys[match['formatted_address']] = match
    return list(addys.values())


def collapse_similar_addresses(matches, parts):
    addys = {}
    for match in sorted(matches, key=lambda x: x['accuracy']):
        key = '-'.join([
            match['address_components'].get(part, '')
            for part in parts
        ])
        addys[key] = match
    return list(addys.values())

def select_most_accurate(matches):
    matches = sorted(matches, key=lambda x: x['accuracy'], reverse=True)
    return matches[0]

def remove_non_matching_streets(matches, key):
    return [
        match for match in matches
        if ' '.join(match['formatted_address'].lower().split(' ')[:2]).startswith(' '.join(key.lower().split(' ')[:2]))
    ]

def lookup(address_locations, key, details, **kwargs):
    if key not in address_locations:
        return None

    address = address_locations[key]
    matches = address['results']
    matches = collapse_identical_addresses(matches)
    matches = filter_by_state(matches, 'VA')
    matches = filter_by_matching_start(matches, key)
    
    if len(matches) > 1:
        matches = filter_by_cities(matches, [
            'Roanoke',
            'Salem',
            'Vinton',
        ])

    if len(matches) > 1:
        suffix_pairs = [
            # sfx, key_ptx
            ('ave', r'\bave\b'),
            ('cir', r'\bcircle\b'),
        ]
        matches = filter_by_matching_suffixes(matches, key, suffix_pairs)
    
    matches = collapse_similar_addresses(matches, ['number', 'street', 'suffix'])
    if len(matches) > 1:
        max_dist = max_distance(matches)
        if max_dist < 800:
            matches = matches[:1]
        else:
            # last ditch
            matches = remove_non_matching_streets(matches, key)
            if len(matches) == 1:
                return matches[0]
            else:
                print('max_dist', max_dist)

    if len(matches) == 0:
        raise ValueError(f'Address not found: {details} at {key}')
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f'Multiple address locations found:{details} at {key} {matches}')


def geocode(df):
    for index, row in df.iterrows():
        addy = lookup(address_locations, key=row['Pickup Address'], details=row['Pickup Address Details'])
        df.loc[index, 'Pickup Lat'] = addy['location']['lat']
        df.loc[index, 'Pickup Lng'] = addy['location']['lng']
        df.loc[index, 'Pickup addy'] = addy['formatted_address']
        
        addy = lookup(address_locations, key=row['Dropoff Address'], details=row['Dropoff Address Details'])
        df.loc[index, 'Dropoff Lat'] = addy['location']['lat']
        df.loc[index, 'Dropoff Lng'] = addy['location']['lng']
        df.loc[index, 'Dropoff addy'] = addy['formatted_address']
    return df


df = geocode(pd.read_csv(metroflex_data / 'metroflex-2025-02-trip-report-cleaned.csv'))
df

In [ ]:
df = geocode(pd.read_csv(metroflex_data / 'metroflex-2025-02-trip-report-cleaned.csv'))
# https://gis.stackexchange.com/questions/478557/heatmap-using-latitude-and-longitude-coordinates

In [ ]:
{
    'max_pickup_lat': df['Pickup Lat'].max(),
    'max_dropoff_lat': df['Dropoff Lat'].max(),
    
    'max_pickup_lng': df['Pickup Lng'].max(),
    'max_dropoff_lng': df['Dropoff Lng'].max(),
    
    'min_pickup_lat': df['Pickup Lat'].min(),
    'min_dropoff_lat': df['Dropoff Lat'].min(),
    
    'min_pickup_lng': df['Pickup Lng'].min(),
    'min_dropoff_lng': df['Dropoff Lng'].min(),
}

In [ ]:
import math

TILE_SIZE = 256

def point_to_pixels(lon, lat, zoom):
    """convert gps coordinates to web mercator"""
    r = math.pow(2, zoom) * TILE_SIZE
    lat = math.radians(lat)

    x = int((lon + 180.0) / 360.0 * r)
    y = int((1.0 - math.log(math.tan(lat) + (1.0 / math.cos(lat))) / math.pi) / 2.0 * r)

    return x, y

In [ ]:

step = .005
lon_min = -80.11
lon_max = -79.86
lons_full = np.arange(lon_min, lon_max + step, step)

lat_max = 37.35
lat_min = 37.21
lats_full = np.arange(lat_min, lat_max - step, -step)

In [ ]:
zoom = 12
x, y = point_to_pixels(lon_min, lat_max, zoom) # top-left corner
x_tiles, y_tiles = int(x / TILE_SIZE), int(y / TILE_SIZE)

In [ ]:
from io import BytesIO
from PIL import Image
import requests

URL = "https://tile.openstreetmap.org/{z}/{x}/{y}.png".format

# format the url
url = URL(x=x_tiles, y=y_tiles, z=zoom)
print(url)

headers = {
    'User-Agent': 'Roanoke Transit Hobby Project - Justin Bangerter',
}
# make the request
with requests.get(url, headers=headers) as resp:
    resp.raise_for_status() # just in case
    img = Image.open(BytesIO(resp.content))


# plot the tile
plt.imshow(img)
plt.show()

In [ ]:
top, bot = lat_max, lat_min
lef, rgt = lon_min, lon_max

zoom = 13
x0, y0 = point_to_pixels(lef, top, zoom)
x1, y1 = point_to_pixels(rgt, bot, zoom)

print(x0, x1, y0, y1)

x0_tile, y0_tile = int(x0 / TILE_SIZE), int(y0 / TILE_SIZE)
x1_tile, y1_tile = math.ceil(x1 / TILE_SIZE), math.ceil(y1 / TILE_SIZE)

tile_ct = (x1_tile - x0_tile) * (y1_tile - y0_tile)
print('tile_ct', tile_ct)
assert tile_ct < 50, "That's too many tiles!"


In [ ]:
from itertools import product

# full size image we'll add tiles to
img = Image.new('RGB', (
    (x1_tile - x0_tile) * TILE_SIZE,
    (y1_tile - y0_tile) * TILE_SIZE))

# loop through every tile inside our bounded box
for x_tile, y_tile in product(range(x0_tile, x1_tile), range(y0_tile, y1_tile)):
    with requests.get(URL(x=x_tile, y=y_tile, z=zoom), headers=headers) as resp:
        resp.raise_for_status() # just in case
        tile_img = Image.open(BytesIO(resp.content))

    # add each tile to the full size image
    img.paste(
        im=tile_img,
        box=((x_tile - x0_tile) * TILE_SIZE, (y_tile - y0_tile) * TILE_SIZE))

plt.imshow(img)
plt.show()

In [ ]:
x, y = x0_tile * TILE_SIZE, y0_tile * TILE_SIZE
print(x0_tile, y0_tile, TILE_SIZE)
print(x, y)
print(x0, x1, y0, y1)
img = img.crop((
    -int(x - x0),  # left
    -int(y - y0),  # top
    -int(x - x1),  # right
    -int(y - y1))) # bottom

plt.imshow(img)
plt.show()

In [ ]:
import matplotlib.colors as colors
import shapely as shp
from shapely.geometry import Point


plt.close()

# make values for a typical coordinate grid with a specified step
step = .005
lons_full = np.arange(-80.11, -79.86 + step, step)
lats_full = np.arange(37.35, 37.21 - step, -step) # latitudes range from 90 to -90 to match array row indices growing from top to bottom

# get values for a special grid, representing central points of cells in the normal grid
halfstep = step/2
lons = lons_full[:-1] + halfstep
lats = lats_full[:-1] - halfstep

# make a 2D array to keep square polygons corresponding to those grid cells
squares = [[None for col in range(len(lons))] for row in range(len(lats))]
squares_shape = np.asarray(squares).shape

# make shapely polygons and store them in that array
for row, col in np.ndindex(squares_shape):
    # longitudes correspond to columns, latitudes to rows
    lon = lons[col]
    lat = lats[row]
    # define vertices of a square polygon
    bottom_left = (lon - halfstep, lat - halfstep)
    bottom_right = (lon + halfstep, lat - halfstep)
    top_right = (lon + halfstep, lat + halfstep)
    top_left = (lon - halfstep, lat + halfstep)
    coords = (bottom_left, bottom_right, top_right, top_left, bottom_left)
    # make a polygon
    squares[row][col] = shp.Polygon(coords)


trip_point_type = 'Dropoff'

# make a numpy array (of the same shape as that 2D polygon array)
# where each element keeps the number pickups in the destination
trip_count = np.zeros(squares_shape)
for index, df_entry in df.iterrows():
    # each row is a set of line strings representing routes
    lat = df_entry[f'{trip_point_type} Lat']
    lng = df_entry[f'{trip_point_type} Lng']
    point = Point(lng, lat)
    # add passenger and guests to count
    for row, col in np.ndindex(squares_shape):
        square = squares[row][col]
        if point.within(square):
            trip_count[row, col] += 1 #+ int(df_entry['Guests'])

# get coastlines and countries
# countries_file = gpd.datasets.get_path('naturalearth_lowres')
# countries_gdf = gpd.read_file(countries_file)

# show everything on one plot
fig, ax = plt.subplots(figsize=(15, 8))
plt.title(f'Distribution of Metroflex {trip_point_type}: Roanoke Metroflex Feb, 2025')
# show coastlines and countries
# countries_gdf.plot(ax=ax, facecolor='none', edgecolor='grey')

# make bounds for 10 sections of a colour bar
# cmap = cm.OrRd
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('', [
    'white',
    'darkblue',
    'darkgreen',
    'yellow',
    'darkorange',
    'darkred',
])

trips_min = trip_count.min()
trips_max = trip_count.max()
bounds = np.linspace(trips_min, trips_max + 1, 11)
norm = colors.BoundaryNorm(bounds, ncolors=cmap.N)

# show squares, whose colour represents the trip count
extent = (min(lons_full), max(lons_full), min(lats_full), max(lats_full))

# request = cimgt.OSM()
# ax = plt.axes(projection=request.crs)
# ax.set_extent(extent)
# ax.add_image(request, 12)

ax.imshow(img, extent=(lef, rgt, bot, top))

im = ax.imshow(trip_count, cmap=cmap, alpha=0.5, extent=extent, norm=norm)
# add a color bar
cb = plt.colorbar(im)
cb.set_label('Number of Trips')
ax.set_xlabel('Longitude (decimal degrees)')
ax.set_ylabel('Latitude (decimal degrees)')

plt.show()